In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Chargement des données
df = pd.read_csv('../data/immobilier_clean.csv')

print(f"Dataset chargé : {df.shape}")

# Visualisation 1 — Prix moyen au m² par commune
prix_commune = df.groupby('Commune')['Prix_m2'].mean().round(0).reset_index()
prix_commune = prix_commune.sort_values('Prix_m2', ascending=False).head(15)

fig1 = px.bar(
    prix_commune,
    x='Commune',
    y='Prix_m2',
    title='Top 15 communes — Prix moyen au m² (Alpes-Maritimes 2025)',
    color='Prix_m2',
    color_continuous_scale='RdYlGn_r',
    text='Prix_m2'
)
fig1.update_traces(texttemplate='%{text:.0f} €', textposition='outside')
fig1.update_layout(
    yaxis_title="Prix moyen au m² (€)",
    xaxis_title="Commune",
    xaxis_tickangle=-45,
    showlegend=False
)
fig1.show()

Dataset chargé : (28903, 9)


In [2]:
# Visualisation 2 — Distribution des prix à Nice, Cannes, Antibes
villes_principales = ['NICE', 'CANNES', 'ANTIBES', 'MENTON']
df_villes = df[df['Commune'].isin(villes_principales)]

fig2 = px.box(
    df_villes,
    x='Commune',
    y='Prix_m2',
    color='Commune',
    title='Distribution des prix au m² — Villes principales (2025)',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig2.update_layout(
    yaxis_title="Prix au m² (€)",
    xaxis_title="Ville",
    showlegend=False
)
fig2.show()

In [3]:
# Filtrer les valeurs aberrantes de prix au m²
df_clean = df[df['Prix_m2'] < 20000].copy()
df_clean = df_clean[df_clean['Prix_m2'] > 1000].copy()

print(f"Après filtrage : {len(df_clean)} transactions")

# Refaire la visualisation
villes_principales = ['NICE', 'CANNES', 'ANTIBES', 'MENTON']
df_villes = df_clean[df_clean['Commune'].isin(villes_principales)]

fig2 = px.box(
    df_villes,
    x='Commune',
    y='Prix_m2',
    color='Commune',
    title='Distribution des prix au m² — Villes principales (2025)',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig2.update_layout(
    yaxis_title="Prix au m² (€)",
    xaxis_title="Ville",
    showlegend=False
)
fig2.show()

# Sauvegarder le dataset ultra-propre
df_clean.to_csv('../data/immobilier_clean.csv', index=False)
print("Dataset mis à jour !")

Après filtrage : 27304 transactions


Dataset mis à jour !


In [4]:
# Visualisation 3 — Prix par type de bien et par ville
fig3 = px.bar(
    df_villes.groupby(['Commune', 'Type local'])['Prix_m2'].mean().round(0).reset_index(),
    x='Commune',
    y='Prix_m2',
    color='Type local',
    barmode='group',
    title='Prix moyen au m² — Appartement vs Maison (2025)',
    color_discrete_map={'Appartement': '#3498DB', 'Maison': '#E74C3C'},
    text='Prix_m2'
)
fig3.update_traces(texttemplate='%{text:.0f} €', textposition='outside')
fig3.update_layout(
    yaxis_title="Prix moyen au m² (€)",
    xaxis_title="Ville",
    legend_title="Type de bien"
)
fig3.show()

In [5]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Préparation des features
df_model = df_clean[['Commune', 'Type local', 'Surface reelle bati', 
                      'Nombre pieces principales', 'Prix_m2']].dropna()

# Encodage des variables catégorielles
le_commune = LabelEncoder()
le_type = LabelEncoder()
df_model = df_model.copy()
df_model['Commune_enc'] = le_commune.fit_transform(df_model['Commune'])
df_model['Type_enc'] = le_type.fit_transform(df_model['Type local'])

# Features et target
X = df_model[['Commune_enc', 'Type_enc', 'Surface reelle bati', 'Nombre pieces principales']]
y = df_model['Prix_m2']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modèle
model = GradientBoostingRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Performance du modèle :")
print(f"MAE : {mae:.0f} €/m²")
print(f"R² : {r2:.3f}")

# Test prédiction
def predire_prix(commune, type_bien, surface, nb_pieces):
    commune_enc = le_commune.transform([commune])[0]
    type_enc = le_type.transform([type_bien])[0]
    prix_m2 = model.predict([[commune_enc, type_enc, surface, nb_pieces]])[0]
    prix_total = prix_m2 * surface
    print(f"\n🏠 Prédiction pour {type_bien} {surface}m² à {commune} ({nb_pieces} pièces)")
    print(f"   Prix estimé au m² : {prix_m2:.0f} €/m²")
    print(f"   Prix total estimé : {prix_total:,.0f} €")

# Exemples
predire_prix('NICE', 'Appartement', 60, 3)
predire_prix('CANNES', 'Maison', 120, 5)
predire_prix('ANTIBES', 'Appartement', 45, 2)

Performance du modèle :
MAE : 1865 €/m²
R² : 0.151

🏠 Prédiction pour Appartement 60m² à NICE (3 pièces)
   Prix estimé au m² : 4940 €/m²
   Prix total estimé : 296,370 €

🏠 Prédiction pour Maison 120m² à CANNES (5 pièces)
   Prix estimé au m² : 7327 €/m²
   Prix total estimé : 879,188 €

🏠 Prédiction pour Appartement 45m² à ANTIBES (2 pièces)
   Prix estimé au m² : 5798 €/m²
   Prix total estimé : 260,888 €


c:\Users\kadia\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\kadia\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\kadia\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(


## Résultats du modèle prédictif

### Performance
- **MAE : 1 865 €/m²** — erreur moyenne de prédiction
- **R² : 0.151** — le modèle explique 15% de la variance

### Limites & améliorations possibles
- Ajouter : proximité mer, étage, année construction, exposition
- Utiliser des données de plusieurs années
- Intégrer les données de géolocalisation précises

### Prédictions exemples
| Bien | Surface | Ville | Prix estimé |
|------|---------|-------|-------------|
| Appartement | 60m² | Nice | 296 370 € |
| Maison | 120m² | Cannes | 879 188 € |
| Appartement | 45m² | Antibes | 260 888 € |